In [4]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Dashboard Components
import dash_leaflet as dl
from dash import dcc, html, dash_table
import plotly.express as px
from dash.dependencies import Input, Output

import base64
import pandas as pd

# CRUD Module
from CRUD_Python_Module import AnimalShelter

JupyterDash.infer_jupyter_proxy_config()

###########################
# Database Connection
###########################

username = "aacuser"
password = "ShelterDogs88!"

db = AnimalShelter(username, password)

df = pd.DataFrame.from_records(db.read({}))
df.drop(columns=['_id'], inplace=True, errors='ignore')

###########################
# Dashboard Layout
###########################

app = JupyterDash(__name__)

# CHANGE THIS IF YOUR LOGO NAME IS DIFFERENT
image_filename = 'Grazioso Salvare Logo.png'

encoded_image = base64.b64encode(
    open(image_filename, 'rb').read()
)

app.layout = html.Div([

    html.Center(
        html.Img(
            src='data:image/png;base64,{}'.format(
                encoded_image.decode()
            ),
            height=120
        )
    ),

    html.Center(html.H1("Grazioso Salvare Dashboard")),
    html.Center(html.H3("Created by Briahna Gerbasi")),

    html.Hr(),

    html.Div([
        html.Label("Select Rescue Type"),

        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'mountain'},
                {'label': 'Disaster or Individual Tracking', 'value': 'disaster'},
                {'label': 'Reset', 'value': 'reset'}
            ],
            value='reset',
            inline=True
        )
    ]),

    html.Hr(),

    dash_table.DataTable(
        id='datatable-id',

        columns=[
            {
                "name": i,
                "id": i,
                "deletable": False,
                "selectable": True
            }
            for i in df.columns
        ],

        data=df.to_dict('records'),

        page_size=10,
        sort_action="native",
        filter_action="native",

        row_selectable="single",
        selected_rows=[0],

        style_table={
            'overflowX': 'auto'
        },

        style_cell={
            'textAlign': 'left',
            'minWidth': '120px',
            'width': '150px',
            'maxWidth': '250px',
            'whiteSpace': 'normal'
        }
    ),

    html.Br(),

    html.Div(
        className='row',
        style={'display': 'flex'},

        children=[

            html.Div(
                id='graph-id',
                style={'width': '50%'}
            ),

            html.Div(
                id='map-id',
                style={'width': '50%'}
            )

        ]
    )

])

#############################################
# Filter Callback
#############################################

@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):

    if filter_type == 'water':

        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "Labrador Retriever Mix",
                    "Chesapeake Bay Retriever",
                    "Newfoundland"
                ]
            }
        }

    elif filter_type == 'mountain':

        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "German Shepherd",
                    "Alaskan Malamute",
                    "Old English Sheepdog",
                    "Siberian Husky",
                    "Rottweiler"
                ]
            }
        }

    elif filter_type == 'disaster':

        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "Doberman Pinscher",
                    "German Shepherd",
                    "Golden Retriever",
                    "Bloodhound",
                    "Rottweiler"
                ]
            }
        }

    else:

        query = {}

    dff = pd.DataFrame.from_records(db.read(query))
    dff.drop(columns=['_id'], inplace=True, errors='ignore')

    return dff.to_dict('records')

#############################################
# Pie Chart
#############################################

@app.callback(
    Output('graph-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data')]
)
def update_graphs(viewData):

    if viewData is None:
        return []

    dff = pd.DataFrame.from_dict(viewData)

    if dff.empty:
        return []

    return [

        dcc.Graph(
            figure=px.pie(
                dff,
                names='breed',
                title='Breed Distribution'
            )
        )

    ]

#############################################
# Highlight Selected Column
#############################################

@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):

    if selected_columns is None:
        return []

    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]

#############################################
# Map Callback
#############################################

@app.callback(
    Output('map-id', 'children'),
    [
        Input('datatable-id', 'derived_virtual_data'),
        Input('datatable-id', 'derived_virtual_selected_rows')
    ]
)
def update_map(viewData, selected_rows):

    if viewData is None:
        return []

    dff = pd.DataFrame.from_dict(viewData)

    if dff.empty:
        return []

    row = 0

    if selected_rows and len(selected_rows) > 0:
        row = selected_rows[0]

    return [

        dl.Map(
            style={
                'width': '100%',
                'height': '500px'
            },

            center=[30.75, -97.48],
            zoom=10,

            children=[

                dl.TileLayer(),

                dl.Marker(

                    position=[
                        dff.iloc[row]['location_lat'],
                        dff.iloc[row]['location_long']
                    ],

                    children=[

                        dl.Tooltip(
                            str(dff.iloc[row]['breed'])
                        ),

                        dl.Popup([
                            html.H3(
                                str(dff.iloc[row]['name'])
                            )
                        ])

                    ]
                )

            ]
        )

    ]

###########################
# Run Dashboard
###########################

app.run_server()

Dash app running on https://plutosurvive-objectroman-3000.codio.io/proxy/8050/
